In [0]:
%pip install pymupdf

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import pymupdf
import re
import os
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pathlib import Path

src = '/Workspace/Shared/HANDSON SANDBOX/downloads'
subfolders = [f for f in Path(src).iterdir() if f.is_dir()]
if not subfolders: raise FileNotFoundError(f"No subfolders found in {src}")
latest_folder = max(subfolders,key=lambda f: datetime.strptime(f.name, "%Y%m%d_%H%M%S"))
source = latest_folder
# print(source)


2026-10-26 10:21:32.902015037 [W:onnxruntime:Default, device_discovery.cc:95 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/ACPI0004:00/MSFT1000:00/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [0]:
X_BOUNDS = [
    0, 65, 140, 210, 280, 350, 420,
    480, 545, 635, 660, 700, 755, 792
]

COLUMN_NAMES = [
    "result_date",
    "patient_raw",
    "facility",
    "location",
    "result",
    "actions_and_plans",
    "comments_assigned_to",
    "flags",
    "notifications",
    "notes",
    "patient_portal",
    "results_processed",
    "photos"
]

DATE_PATTERN = re.compile(r"^\d{2}/\d{2}/\d{4}$")
DOB_PATTERN = re.compile(r"\bDOB\s+(\d{2}/\d{2}/\d{4})\b")
MRN_PATTERN = re.compile(r"\bMRN\s+([A-Z]{2,}-\d+)\b")


# --------------------------------------------------
# TEXT HELPERS
# --------------------------------------------------

def clean_text(value):
    """Normalize whitespace without removing meaningful punctuation."""
    if not value:
        return None

    value = re.sub(r"\s+", " ", value).strip()
    return value or None


def unique_join(values, separator=" "):
    """Join multiline values while removing consecutive duplicates."""
    cleaned = [clean_text(v) for v in values]
    cleaned = [v for v in cleaned if v]

    output = []
    for value in cleaned:
        if not output or output[-1] != value:
            output.append(value)

    return separator.join(output) if output else None


def words_to_lines(words, tolerance=2.5):
    """
    Group PDF words by approximate vertical position.
    Each word is represented as:
    (x0, y0, x1, y1, text, ...)
    """
    lines = []

    for word in sorted(words, key=lambda w: (w[1], w[0])):
        x0, y0, x1, y1, text = word[:5]

        target = None
        for line in lines:
            if abs(line["y"] - y0) <= tolerance:
                target = line
                break

        if target is None:
            target = {"y": y0, "words": []}
            lines.append(target)

        target["words"].append(word)

    for line in lines:
        line["words"].sort(key=lambda w: w[0])

    return sorted(lines, key=lambda line: line["y"])


def extract_column_text(words, x_left, x_right):
    """Extract words that fall inside a specified column."""
    selected = [
        word for word in words
        if word[0] >= x_left and word[0] < x_right
    ]

    lines = words_to_lines(selected)

    return unique_join([
        " ".join(word[4] for word in line["words"])
        for line in lines
    ])

def extract_pdf_records(pdf_path):
    records = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):

            words = page.get_text("words", sort=True)

            # Remove page headings and footer content.
            # The table body lies approximately between y=65 and y=570.
            words = [
                word for word in words
                if 65 <= word[1] < 570
            ]

            # Each record begins with a date in the first column.
            anchors = []

            for word in words:
                x0, y0, x1, y1, text = word[:5]

                if (
                    x0 < 65
                    and DATE_PATTERN.fullmatch(text.strip())
                ):
                    anchors.append((y0, text.strip()))

            anchors.sort(key=lambda item: item[0])

            for index, (start_y, date_text) in enumerate(anchors):

                # Stop at the next record's date.
                # The last record ends at the table-body boundary.
                end_y = (
                    anchors[index + 1][0]
                    if index + 1 < len(anchors)
                    else 570
                )

                record_words = [
                    word for word in words
                    if start_y - 1 <= word[1] < end_y
                ]

                row = {}

                for i, column in enumerate(COLUMN_NAMES):
                    row[column] = extract_column_text(
                        record_words,
                        X_BOUNDS[i],
                        X_BOUNDS[i + 1]
                    )

                row["result_date"] = date_text
                row["source_page"] = str(page_number)

                # Extract patient identifiers from multiline details.
                patient_raw = row.get("patient_raw") or ""

                dob_match = DOB_PATTERN.search(patient_raw)
                mrn_match = MRN_PATTERN.search(patient_raw)

                row["dob"] = (
                    dob_match.group(1) if dob_match else None
                )
                row["mrn"] = (
                    mrn_match.group(1) if mrn_match else None
                )

                # Remove DOB/MRN text to isolate the patient name.
                patient_name = DOB_PATTERN.sub("", patient_raw)
                patient_name = MRN_PATTERN.sub("", patient_name)
                row["patient_name"] = clean_text(patient_name)

                # Keep every output field as a string initially.
                records.append(row)

    return records

In [0]:
for pdf in os.listdir(source):
    if not pdf.endswith(".pdf"): continue
    PDF_PATH = os.path.join(source, pdf)
    # print(PDF_PATH)
    records = extract_pdf_records(PDF_PATH)
    schema = StructType([
        StructField("result_date", StringType(), True),
        StructField("patient_name", StringType(), True),
        StructField("dob", StringType(), True),
        StructField("mrn", StringType(), True),
        StructField("facility", StringType(), True),
        StructField("location", StringType(), True),
        StructField("result", StringType(), True),
        StructField("actions_and_plans", StringType(), True),
        StructField("comments_assigned_to", StringType(), True),
        StructField("flags", StringType(), True),
        StructField("notifications", StringType(), True),
        StructField("notes", StringType(), True),
        StructField("patient_portal", StringType(), True),
        StructField("results_processed", StringType(), True),
        StructField("photos", StringType(), True),
        StructField("source_page", StringType(), True)
    ])

    # Select fields in schema order.
    spark_rows = [
        tuple(record.get(field.name) for field in schema.fields)
        for record in records
    ]

    pathology_df = spark.createDataFrame(spark_rows, schema)
    pathology_df.write.mode("append").saveAsTable(f"us_scdb_phi_dev.comm_bronze.test_{pdf[15:-23]}")
    print(f'{pdf[15:-23]}')
                                                  

user_a35
user_a20
user_a16
user_a0
user_a4
user_a44
user_a49
user_a2
user_a27
user_a39
user_a4
user_a23
user_a50
user_a30
user_a13
user_a40
user_a6
user_a15
user_a25
user_a44
user_a19
user_a50
user_a17
user_a31
user_a32
user_a38
user_a12
user_a13
user_a42
user_a45
user_a17
user_a20
user_a14
user_a19
user_a50
user_a10
user_a23
user_a41
user_a22
user_a30
user_a14
user_a46
user_a25
user_a37
user_a8
user_a21
user_a24
user_a36
user_a49
user_a21
user_a27
user_a33
user_a12
user_a42
user_a24
user_a6
user_a29
user_a47
user_a48
user_a30
user_a33
user_a3
user_a8
user_a9
user_a0
user_a26
user_a48
user_a24
user_a29
user_a41
user_a2
user_a49
user_a11
user_a41
user_a16
user_a0
user_a1
user_a7
user_a32
user_a2
user_a4
user_a21
user_a15
user_a25
user_a19
user_a28
user_a16
user_a23
user_a26
user_a36
user_a35
user_a5
user_a10
user_a12
user_a37
user_a45
user_a8
user_a18
user_a38
user_a44
user_a34
user_a35
user_a36
user_a46
user_a14
user_a40
user_a42
user_a43
user_a45
user_a47
user_a38
user_a33
user_a34
us